# Principles of MRI Lab: Image Formation

This lab explores MRI image formation, including slice selection and frequency/phase encoding.

Borrow from Resonint labs - imaging objects  (probably not projection reconstruction)

>-------------------------------------------------------------------------------------------------------------------------------------------------------
> #### **Setup Task: Run the Notebook**
> 
> 1. Edit the cell below to set the `LAB_USER_NAME` variable to your name
> 2. Click **Run->Run All Cells** in the in top menu bar of jupyterlab
> 3. Open the Table of Contents side-bar on the left edge of jupyterlab to aid in navigation
> 
> -------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
LAB_USER_NAME = 'Student'

**Important**: To initialise this notebook, edit the cell above to set `LAB_USER_NAME` to your name, then click **Run->Run All Cells** in the top menu bar.

In [ ]:
import panel as pn
import sys
import os
import yaml
import numpy as np

from matipo import Sequence, SEQUENCE_DIR, GLOBALS_DIR, DATA_DIR, Unit
from matipo.util.decimation import decimate
from matipo.util.fft import fft_reconstruction
from matipo.util.etl import deinterlace

from matipo.experiment.base_experiment import BaseExperiment, auto_inputs, PlotInterface
from matipo.experiment.models import PLOT_COLORS, SITickFormatter, SIHoverFormatter

from matipo.experiment.plots import SignalPlot, SpectrumPlot, ImagePlot, Image1DPlot, ComplexDataLinePlot

pn.extension(inline=True)

WORKSPACE = os.path.join(LAB_USER_NAME)
LAB_DIR = os.path.join('/home/data/PrinciplesOfMRI', LAB_USER_NAME)
os.makedirs(LAB_DIR, exist_ok=True)
print('Data will be saved to', LAB_DIR)

GAMMA_BAR = 42.58e6
RESOLUTION = 64
FOV = 12e-3 # m
AXIS_VECTORS = {
    'X': np.array([1,0,0], dtype=float),
    'Y': np.array([0,1,0], dtype=float),
    'Z': np.array([0,0,1], dtype=float)
}

try:
    with open(GLOBALS_DIR+'hardpulse_90.yaml', 'r') as f:
        data = yaml.load(f, Loader=yaml.SafeLoader)
        AREA_90 = data['a_90']*data['t_90']
except IOError:
    print('Unable to load pulse calibration, using default value')
    AREA_90 = 0.3*32e-6
    
def estimate_a_90(pulse_shape, pulse_width):
    shape_area = np.mean(pulse_shape)*pulse_width
    return min(1, abs(AREA_90/shape_area))

## Fundamentals

Copy some key points from book

### Slice Selection

RF pulse slice profile with shim sample

Change pulse time
Change pulse shape
Change gradient amplitude
Change gradient direction
Choose Refocusing gradient area
Change RF pulse frequency offest

### Frequency Encoding

Show data and FT

Point object?  with narrow slice select?

Point objects with multiple slices?

Rect object - with slice select

Change readout gradient amplitude
    duration
Sampling bandwidth

### 2D imaging and K-space

(maybe from 2D RARE?)

Show k-space trajectory

Show 2D k-space
real/imaginary

Show 2D image
real/imaginary

Options to manipulate pulse sequence parameters (Gxr, Gyi, Gyp, NFE, NPE, Tread, Tpe, BW)?

## Experiments

The tradeoff between resolution, scan time and SNR can be explored with the following imaging apps.

### Setup

Position the shim phantom of the center of the coil.  Make sure to calibrate the system (e.g. Find Frequency) prior to imaging.

## Slice Selection

> -------------------------------------------------------------------------------------------------------------------------------------------------------
> #### **Visualising Sinc Pulse Design**
>The experiment below is designed to help you visualise how changing the parameters of a soft pulse changes the profile in the frequency domain. The RF pulse you design will be used for the following experiments.
> 1. Experiment with the three soft pulse parameters and observe how the shape of the RF-pulse and slice profile changes. 
> 2. **Question:** How does changing the lobe width affect the spectral bandwidth?
> 3. **Question:** How does adding more lobes improve the shape of the slice profile? What is the trade-off regarding the overall pulse width?
> 4. **Question:** What happens when apodization is too large? Pick a value that looks like a good compromise.
> -------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
from custom_RF import CustomRF

class SincRF(CustomRF):
    
    def setup(self):
        super().setup()
        self.title = "Sinc RF Pulse Design"
        self.workspace = WORKSPACE
        
        self.inputs = {
            'lobe_width': pn.widgets.FloatInput(name="Lobe Width (μs)", start=50, end=400, step=50, value=100, width=80),
            'n_lobe': pn.widgets.FloatInput(name="Lobes", start=0, end=10, step=1, value=3, width=80),
            'apodization': pn.widgets.FloatInput(name="Apodization", start=0, end=100, step=1, value=0, width=80)
        }
        
        # auto frequency range based on max bandwidth
        BW_max = 1e-6*self.inputs['lobe_width'].start
        self.plots['spectrum'].fig.x_range.start = -4/BW_max
        self.plots['spectrum'].fig.x_range.end = 4/BW_max
        
        # update plots when any value is changed
        self.inputs['lobe_width'].param.watch(self.pulse_settings_handler, 'value_throttled')
        self.inputs['n_lobe'].param.watch(self.pulse_settings_handler, 'value_throttled')
        self.inputs['apodization'].param.watch(self.pulse_settings_handler, 'value_throttled')

        # manually trigger for first update
        self.inputs['lobe_width'].param.trigger('value_throttled')
    
    
    def calc_pulse_shape(self):
        width = max(1, self.inputs['n_lobe'].value)*self.inputs['lobe_width'].value*1e-6 # max(1, ...) to prevent width=0 when n_lobes=0
        N = int(width / 1e-6)
        if N > 1000:
            N = 1000
        dt = width / N
        pts = np.sinc(np.linspace(-self.inputs['n_lobe'].value, self.inputs['n_lobe'].value,N))
        pts*=np.exp(-self.inputs['apodization'].value*(np.linspace(-1, 1, N)**2))
        return width, pts
        

VisualizeRFExperiment = SincRF(state_id='lab3')
VisualizeRFExperiment()

> -------------------------------------------------------------------------------------------------------------------------------------------------------
> #### **1D Slice Profile**
> With the Sinc pulse designed, we can now use it to create a 1D image of the slice profile.
> You will need Equation 1: $$\Delta z = \frac{\Delta F}{\bar\gamma \cdot G_{ss}}$$ where $\bar\gamma = 42.58 \times 10^{6} \ \mathrm{Hz}/\mathrm{T}$ for the hydrogen nucleus.
> 1. **Question:** Based on the bandwidth ($\Delta$ F) of the Sinc pulse you have designed and a slice select gradient $G_{ss}=100 \ \mathrm{mT}/\mathrm{m}$, calculate the expected thickness of the slice profile.
> 2. Insert the **Shim Sample** (make sure to centre the sample using the depth gauge), set the slice select gradient, and run the experiment.
> 3. **Question:** How does the width of the slice profile compare to your predicted width?
> 4. Try changing the flip angle to 30 and 180 and compare the resulting spectrum.
> 5. **Question:** At what flip angle (out of 30, 90, or 180) does the signal spectrum look most similar to the RF pulse spectrum shown above?
> 1. Try changing the frequency offset of the RF pulse.
> 1. **Question:** How does the slice profile change when the frequency offset is changed?
> -------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
class SliceProfile(BaseExperiment):
    
    def setup(self):
        self.title = "Sinc RF Pulse Slice Profile Measurement"
        self.workspace = WORKSPACE
        self.seq = Sequence('custom_pulse_SE.py')
        self.plots = {
            'signal': SignalPlot(),
            'image': Image1DPlot()
        }
        
        with open(os.path.join(GLOBALS_DIR, 'gradient_calibration.yaml'), 'r') as f:
            data = yaml.load(f, Loader=yaml.SafeLoader)
            self.G_CAL = 1/(GAMMA_BAR*data['gradient_calibration']) # convert
            self.log.debug(f'G_CAL: {self.G_CAL}')
        
        self.inputs={
            'freq_offset':pn.widgets.FloatInput(name="RF Pulse Frequency Offset (kHz)", start=-30, end=30, step=1, value=0, width=200, disabled=False),
            'GSS':pn.widgets.FloatInput(name="Slice Select Gradient (mT/m)", start=0, end=150, step=10, value=100, width=200, disabled=False),
            'flip_angle':pn.widgets.FloatInput(name="Target Flip Angle (degrees)", start=10, end=360, step=10, value=90, width=200, disabled=False)
        }
        self.DEC = 4
        self.G_IMAGE = 0.1 # T/m
    
    def update_par(self):
        self.seq.setpar(
            a_90=self.inputs['flip_angle'].value/90*estimate_a_90(VisualizeRFExperiment.pulse_shape,VisualizeRFExperiment.pulse_width),
            t_90=VisualizeRFExperiment.pulse_width,
            shape_90=VisualizeRFExperiment.pulse_shape,
            f_slice_offset=1e3*self.inputs['freq_offset'].value,
            g_slice=(0,0,-1e-3*self.inputs['GSS'].value/self.G_CAL),
            g_read=(0,0,self.G_IMAGE/self.G_CAL),
            n_scans=4,
            n_samples=200*self.DEC,
            t_dw=20e-6/self.DEC, # using a long dwell time for narrow bandwith to more easily see the spectrum shape
            t_end=0.5
        )
    
    async def update_plots(self, *args, **kwargs):
        await self.seq.fetch_data()
        self.plots['signal'].update(seqdata=self.seq.data, t_dw=self.seq.par.t_dw)
        self.plots['image'].update(
            seqdata=self.seq.data,
            t_dw=self.seq.par.t_dw,
            g_read_mag=self.G_CAL*np.linalg.norm(self.seq.par.g_read),
            dec=self.DEC
        )
            

SliceProfileExperiment = SliceProfile(state_id='sinc_slice_profile')
SliceProfileExperiment()

## Frequency Encoding

>---
>#### **Frequency Encoding**
>1. Insert a phantom at the correct depth and start the experiment below with **"Run Loop"**.
>2. Adjust the read gradient to 25%.  
>1. **Question:** What do you see in the spectrum?  Does this match your expectations based on the phantom you inserted?
>1. Trying increasing and decreasing the read gradient strength.
>1. **Question:** How does the spectrum change when the read gradient is changed?
>1. Adjust the read gradient direction and note how the X and Y read gradients components change.
>1. **Question:** How does the spectrum change when the read gradient direction is changed by 90-degrees?  What does this correspond to in the phantom?
>1. **Question:** How does the spectrum change when the read gradient direction is changed by 180-degrees?  What does this correspond to in the phantom?
>1. Repeat the experiment with a different phantom and note how the spectrum changes with manipulation of the read gradient strength and direction.
>---

In [ ]:
class FrequencyEncoding(BaseExperiment):
    
    def setup(self):
        self.enable_runloop = True
        self.title = "Frequency Encoding Experiment"
        self.workspace = WORKSPACE
        self.seq = Sequence('SE.py')
        self.plots = {
            'signal': SignalPlot(),
            'spectrum': SpectrumPlot(show_magnitude=True)
        }
        
        self.inputs = auto_inputs(self.seq, {
        })
        
        self.inputs['read_grad_mag'] = pn.widgets.IntInput(name='Read Gradient Magnitude (%)', start=0, end=100, value=0, width=200)
        self.inputs['read_grad_dir'] = pn.widgets.IntInput(name='Read Gradient Direction (degrees)', start=0, end=360, step=10, value=0, width=200)
        self.inputs['read_x_grad'] = pn.widgets.IntInput(name='Read Gradient X Component (%)', start=-100, end=100, value=0, width=200, disabled=True)
        self.inputs['read_y_grad'] = pn.widgets.IntInput(name='Read Gradient Y Component (%)', start=-100, end=100, value=0, width=200, disabled=True)
        
        def exp3_read_grad_cb(*events):
            self.inputs['read_x_grad'].value = int(self.inputs['read_grad_mag'].value*np.cos((np.pi/180)*self.inputs['read_grad_dir'].value))
            self.inputs['read_y_grad'].value = int(self.inputs['read_grad_mag'].value*np.sin((np.pi/180)*self.inputs['read_grad_dir'].value))

        self.inputs['read_grad_mag'].param.watch(exp3_read_grad_cb, ['value'], onlychanged=True)
        self.inputs['read_grad_dir'].param.watch(exp3_read_grad_cb, ['value'], onlychanged=True)
        
        self.DEC = 4
    
    def update_par(self):
        self.seq.setpar(
            g_read=(0.01*self.inputs['read_grad_mag'].value*np.cos((np.pi/180)*self.inputs['read_grad_dir'].value), 0.01*self.inputs['read_grad_mag'].value*np.sin((np.pi/180)*self.inputs['read_grad_dir'].value), 0),
            t_echo=0.012,
            n_scans=2,
            n_samples=100,
            t_dw=20e-6, # using a long dwell time for narrow bandwith to more easily see the spectrum shape
            t_end=0.1
        )
        
        # supersample for postprocess decimation
        self.seq.setpar(
            n_samples=self.seq.par.n_samples*self.DEC,
            t_dw=self.seq.par.t_dw/self.DEC
        )
    
    async def update_plots(self, *args, **kwargs):
        await self.seq.fetch_data()
        self.plots['signal'].update(seqdata=self.seq.data, t_dw=self.seq.par.t_dw)
        self.plots['spectrum'].update(seqdata=decimate(self.seq.data, self.DEC), t_dw=self.seq.par.t_dw, t0='auto', fast=True)
            

FrequencyEncodingExperiment = FrequencyEncoding(state_id='fe_experiment')
FrequencyEncodingExperiment()